<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-10-tuning-and-evaluation/lesson-10.2-context-caching/notebooks/GCP_Capstone_10.2_ContextCaching.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10.2 Context Caching — The Cache the API Always Carried, Finally Created
**Netsetos GenAI Engineering — GCP Capstone** · Module 10 · rebuilt on the live lane, 9 September 2026

The API has carried a per-tenant cache manager since 5 September and read it on every request; nothing ever created a cache, so every answer paid full price for its context. This lesson creates one through the kit's own manager - the tenant's documents as the pack, the generator's rules as the instruction, the corpus manifest's hash as the version - and watches the lane use it on the next question, in the usage row's `cached_tokens`. Then the other three verbs, break-even from measured numbers, the versioning rule, and the tiered pattern mapped onto tenants.


## Setup


In [ ]:
!pip install -q google-genai==2.22.0 google-cloud-firestore==2.30.0 google-cloud-storage==3.13.1 fastapi==0.141.1 pydantic-settings==2.15.0 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp"   # the kit: deploy/shared is the tool layer every lesson on the lane imports
BRANCH     = "feat/lesson-4.8-live-evals"        # the demo branch; main is behind it

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google import genai
from google.genai import types

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
API_URL       = f"https://documind-api-{NUMBER}.{REGION}.run.app"
UPLOAD_BUCKET = f"{PROJECT_ID}-uploads"     # storage.tf: the bucket eventarc.tf watches - the corpus, media included
MEDIA_BUCKET  = f"{PROJECT_ID}-media"       # storage.tf: generated assets, 30-day lifecycle (a cache, not a record)
DATASETS      = f"{PROJECT_ID}-datasets"    # storage.tf (Module 10): the frozen tuning dataset and 10.5's GGUF
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": API_URL,
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MEMBER_SA   = os.environ["DOCUMIND_IMPERSONATE_SA"]
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"   # IAM admits it, no roster does (4.8, 7.2)

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - the contract gate fails a paste.
gen = genai.Client(enterprise=True, project=PROJECT_ID, location="global")   # every generate_content in this lesson
os.environ["GENERATOR_MODEL"] = "gemini-3.6-flash"   # the cache belongs to a model: the API's, unless a candidate is running

print("kit:", KIT, "| API:", API_URL, "| datasets:", f"gs://{DATASETS}/sft/")


## Cell 1: The API and its usage rows


In [ ]:
import json, requests, time, subprocess, datetime
from google.cloud import storage

# THE API, CALLED THE WAY THE UI CALLS IT: one ID token per request, minted AS the roster member,
# audience = the API (7.3's hour-long fuse never arms). The kit mints it (documind_tools._id_token).
def api(path: str, body: dict | None = None, base: str | None = None, timeout: int = 120) -> tuple[int, dict | str]:
    """POST one API route (or a candidate revision's, with base=) as documind-ui-sa. Returns (status, json-or-text)."""
    url = (base or API_URL).rstrip("/")
    r = requests.post(f"{url}{path}", json=body,
                      headers={"Authorization": f"Bearer {documind_tools._id_token(url)}"}, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

# The usage rows the API logs - the ONE shape every observability consumer reads (12.3, tenant_daily). On
# the lean lane they live in Cloud Logging; this reads the last few for a surface, newest first.
def usage_rows(minutes: int = 15, limit: int = 20, event: str = "query") -> list[dict]:
    since = (datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(minutes=minutes)).strftime("%Y-%m-%dT%H:%M:%SZ")
    r = subprocess.run(["gcloud", "logging", "read",
                        f'resource.type="cloud_run_revision" AND resource.labels.service_name="documind-api" '
                        f'AND jsonPayload.event="{event}" AND timestamp>="{since}"',
                        "--project", PROJECT_ID, "--limit", str(limit), "--format=json"], capture_output=True, text=True)
    try:
        return [e["jsonPayload"] for e in json.loads(r.stdout or "[]")]
    except ValueError:
        return []

gcs = storage.Client(project=PROJECT_ID)

def gcs_text(uri: str) -> str:
    bucket, _, name = uri.removeprefix("gs://").partition("/")
    return gcs.bucket(bucket).blob(name).download_as_text()

sys.path.insert(0, f"{KIT}/deploy/evals")               # the kit's builders and judges: make_trainset, judge, tune, run_eval
sys.path.insert(0, f"{KIT}/deploy/services/rag-api")    # the API's own modules: cache_manager, router, breakers, cost
print("helpers: api(), usage_rows(), gcs_text(); the kit's evals/ and rag-api/ on sys.path")


## Cell 2: Implicit caching, already happening
Ten questions, then the usage rows: the repeated SYSTEM prefix is cached for free, and it is small.


In [ ]:
# IMPLICIT CACHING, ALREADY HAPPENING. Every request the API builds starts with the same prefix - the
# generator's SYSTEM - and Gemini caches a repeated prefix on its own; the usage row records what it
# cached (cached_tokens). Ten questions through retrieve(), then the rows: the prefix is short, so the
# hits are small, and that is exactly why the explicit cache below exists.
QUESTIONS = ["What is the notice period for a confirmed E3?", "What is the per-trip cap on domestic travel?",
             "After how many years of continuous service does gratuity become payable?", "What does the Code on Wages define as wages?",
             "What is the maximum period of maternity benefit?", "How is overtime paid under the OSH Code?",
             "What notice does the ACME MSA require to terminate for convenience?", "What is the total payable on invoice INV-2026-0412?",
             "Which region's revenue declined in FY2026?", "What is the leave encashment cap?"]
for q in QUESTIONS:
    documind_tools.retrieve(q, tenant_id=TENANT, brain="direct")
time.sleep(20)                                                     # logs lag
rows = usage_rows(minutes=10, limit=12)
print(f"{'tokens_in':>9} {'cached':>7} {'model':22} question-ish")
for r in rows[:10]:
    print(f"{r.get('tokens_in', 0):>9} {r.get('cached_tokens', 0):>7} {str(r.get('model'))[:22]:22} {r.get('surface')} / {r.get('brain')}")
BEFORE = [r.get("cached_tokens", 0) for r in rows[:10]]
assert rows, "no usage rows in the last ten minutes - the logging read lags; re-run in a minute"
print("\nthe SYSTEM prefix is a few hundred tokens: implicit hits are small change. The corpus is not.")


## Cell 3: The explicit cache, through the kit's manager
`make cache TENANT=acme` in a notebook: the pack, the instruction, the version - and where the cache landed.


In [ ]:
from cache_manager import TenantCacheManager, MIN_CACHE_TOKENS
from cache_admin import pack_for
from generator import SYSTEM

# THE EXPLICIT CACHE, THROUGH THE KIT'S OWN MANAGER. cache_manager.py has been in the API image since
# 5 September and generator.py has read it on every request - and nothing ever CREATED a cache, so every
# answer paid full price for its context. This cell is `make cache TENANT=acme` in a notebook: the tenant's
# synthetic documents as the pack (the handbook, the MSA, the invoice, the report, the town hall
# transcript), the generator's SYSTEM as the instruction, the corpus manifest's hash as the version.
# The cache belongs to ONE model: the manager records it, and the generator attaches it only when the
# request's model matches.
pack, version = pack_for(TENANT, f"{KIT}/deploy/evals/corpus")
tokens = gen.models.count_tokens(model="gemini-3.6-flash", contents=pack).total_tokens
print(f"pack: {len(pack):,} chars, {tokens:,} tokens (minimum for an explicit cache on the Gemini 3 family: {MIN_CACHE_TOKENS:,}) | corpus {version}")
assert tokens >= MIN_CACHE_TOKENS

mgr = TenantCacheManager(PROJECT_ID)
rec = mgr.create(TENANT, SYSTEM, pack, ttl_s=3600, version=version)
print("cache    :", rec["cache_name"])
print("location :", rec["location"], "<- global or regional: the answer to the question CLAUDE.md has carried since 4 September")
print("model    :", rec["model"], "| tokens:", rec["tokens"], "| expires:", rec["expire_time"])
assert rec["tokens"] >= MIN_CACHE_TOKENS and rec["model"] == "gemini-3.6-flash"


## Cell 4: The lane uses it on the next question
Nothing redeployed. The manager's Firestore record is the switch, and `cached_tokens` shows the pack.


In [ ]:
# THE LANE USES IT ON THE NEXT QUESTION. generator._cache_kwargs() reads tenant_caches/acme from Firestore
# and splats cached_content into the request. Nothing was redeployed: the manager's record IS the switch.
# The usage row's cached_tokens now carries the pack - the 90% rate on ~40,000 tokens, every question.
for q in QUESTIONS[:5]:
    documind_tools.retrieve(q, tenant_id=TENANT, brain="direct")
time.sleep(20)
after = [r.get("cached_tokens", 0) for r in usage_rows(minutes=3, limit=5)]
print("cached_tokens before:", BEFORE[:5])
print("cached_tokens after :", after)
assert after and max(after) >= MIN_CACHE_TOKENS, "the API did not attach the cache - is GENERATOR_MODEL on the API the cache's model? (a candidate revision would not be)"
print("\nthe difference is the pack: every question now pays a tenth for the context it always carried")


## Cell 5: The record, and the other verbs


In [ ]:
# THE RECORD, AND THE THREE OTHER VERBS. The manager keeps one Firestore document per tenant - name,
# location, model, tokens, expiry, version - so the API never lists caches and never guesses. get() returns
# nothing for a cache about to expire (the request would fail); refresh() extends the TTL; delete() removes
# both the cache and the record. Explicit caches have no maximum TTL, so every one is created with one.
print("get     :", {k: v for k, v in (mgr.get(TENANT) or {}).items() if k in ("cache_name", "location", "tokens", "version")})
r2 = mgr.refresh(TENANT, ttl_s=7200)
print("refresh :", r2["expire_time"], "(two hours from now)")
print("\nleaving it in place for the session: the TTL bounds the bill. `make cache CACHE_OP=delete` removes it.")


## Cell 6: Break-even, with the lane's numbers


In [ ]:
# BREAK-EVEN, WITH THE LANE'S NUMBERS. The cache costs storage per token-hour and reads at a tenth of the
# input rate; it pays off when the questions per hour cross a line that depends only on the prices. The
# inputs here are measured: the pack's tokens from the cache record, the questions per hour from the
# usage rows of the last hour.
def break_even(cache_tokens: int, queries_per_hour: float, session_hours: int = 1, model: str = "gemini-3.6-flash") -> dict:
    prices = {"gemini-3.6-flash": {"std": 1.50, "cached": 0.15, "storage": 1.00},
              "gemini-3.1-flash-lite": {"std": 0.25, "cached": 0.025, "storage": 1.00},
              "gemini-3.1-pro-preview": {"std": 2.00, "cached": 0.20, "storage": 4.50}}       # USD per 1M tokens (-hour for storage)
    pr = prices[model]
    m = cache_tokens / 1_000_000
    n = queries_per_hour * session_hours
    no_cache = n * m * pr["std"]
    with_cache = m * pr["std"] + n * m * pr["cached"] + session_hours * m * pr["storage"]
    return {"no_cache_usd": round(no_cache, 4), "with_cache_usd": round(with_cache, 4),
            "break_even_queries_per_hour": round(pr["storage"] / (pr["std"] - pr["cached"]), 2)}

qph = len(usage_rows(minutes=60, limit=500))
print("questions in the last hour:", qph, "| pack tokens:", rec["tokens"])
for hours in (1, 8, 24):
    print(f"  {hours:>2} h:", break_even(rec["tokens"], max(qph, 1), hours))
print("\nbelow the break-even line the cache costs more than it saves; a demo's ten questions an hour is above it")


## Cell 7: The corpus manifest is the cache version


In [ ]:
# THE CORPUS MANIFEST IS THE CACHE VERSION. A cache of last week's handbook answers this week's questions
# with last week's clauses, and nothing raises. The manager records the corpus manifest's hash at create
# time; a re-ingest changes the manifest, the hashes differ, and the rule is: recreate, never refresh.
current = pack_for(TENANT, f"{KIT}/deploy/evals/corpus")[1]
stored = (mgr.get(TENANT) or {}).get("version")
print("stored version :", stored)
print("corpus now     :", current)
assert stored == current, "the corpus changed since the cache was made: make cache again"

def cache_is_current(mgr, tenant, version) -> bool:
    rec = mgr.get(tenant)
    return bool(rec) and rec.get("version") == version

print("current?", cache_is_current(mgr, TENANT, current))
print("after a corpus change?", cache_is_current(mgr, TENANT, "a1b2c3d4e5f6"), "<- recreate, do not refresh")


## Cell 8: The tiered pattern, on tenants


In [ ]:
# THE TIERED PATTERN, ON TENANTS. 10.2 used to build an org cache and per-user caches in a notebook class.
# On the lane the tenant IS the organisation, the manager keeps one cache per tenant, and the request's
# tenant picks it - so the tiering is the roster's, not a second registry. Per-user caches would be a
# second key on the same manager, and the first question is whether any user's documents are ever
# large enough (4,096 tokens) to be worth one.
print(f"{'tier':12} {'key':14} {'where it lives':32} {'who creates it'}")
for tier, key, where, who in (("tenant", "tenant_id", "tenant_caches/<tenant> (Firestore)", "make cache TENANT=<tenant>"),
                              ("per user", "tenant:user", "not built - below the minimum for most users", "the day a user's documents exceed 4,096 tokens"),
                              ("implicit", "the prefix", "Gemini, automatically", "nobody - the SYSTEM prefix repeats on every request")):
    print(f"{tier:12} {key:14} {where:32} {who}")


## Where this goes
- **10.3** gives the router the same treatment: code the image carried, put on the request path behind a flag.
- **12.6** owns the refresh on the full profile (a Scheduler job); on lean, `make cache CACHE_OP=refresh` is the hand.

## ✅ Lesson 10.2 complete
- ✅ Implicit hits read off the API's own usage rows
- ✅ An explicit cache created through the kit's manager: the pack, the rules, the version, and where it landed
- ✅ The lane using it on the next question, with no redeploy
- ✅ get, refresh, delete against the Firestore record; break-even from measured numbers
- ✅ The corpus manifest as the cache version; tiering as the roster's
